# ACV-LQN - OpenCV Basic - TH01

Notebook này được tổ chức lại đúng yêu cầu: mỗi mục lớn có 3 code cell chính:

1. **CELL THỦ CÔNG** - tự cài đặt bằng NumPy và thao tác pixel.
2. **CELL TỰ ĐỘNG / THƯ VIỆN** - dùng hàm có sẵn của OpenCV.
3. **CELL SO SÁNH** - so sánh hình ảnh, sai số, thời gian chạy và bộ nhớ xấp xỉ.

Ảnh đầu vào được tải bằng `wget` từ URL Lenna mẫu. Nếu môi trường không có Internet, notebook tự dùng ảnh dự phòng trong thư mục `data/` để vẫn chạy được.

## 0. Tải ảnh mẫu bằng wget

Cell này tải ảnh từ Internet theo đúng yêu cầu. Khi chạy trên Google Colab hoặc máy có Internet, file `data/Lenna.jpg` sẽ được tạo tự động.

In [ ]:
# Tạo thư mục data để chứa ảnh đầu vào và outputs để lưu kết quả xử lý.
!mkdir -p data outputs

# Tải ảnh Lenna mẫu bằng wget. Tham số -O đặt tên file đầu ra cố định.
!wget -q -O data/Lenna.jpg http://www.ess.ic.kanagawa-it.ac.jp/std_img/colorimage/Lenna.jpg

# In thông báo để người chạy notebook biết file nào đang được tải.
print("Đã thử tải ảnh từ Internet vào: data/Lenna.jpg")

## 0.1. Import thư viện và hàm tiện ích chung

Phần này chỉ chuẩn bị môi trường, không tính là một mục xử lý ảnh chính. Các mục xử lý ảnh bên dưới đều theo đúng khuôn 3 cell: thủ công, tự động, so sánh.

In [ ]:
# Path giúp quản lý đường dẫn file rõ ràng hơn so với chuỗi thông thường.
from pathlib import Path

# time dùng để đo thời gian chạy của từng cách xử lý.
import time

# tracemalloc dùng để đo bộ nhớ Python cấp phát trong lúc chạy hàm.
# Lưu ý: bộ nhớ cấp phát sâu trong thư viện C/C++ như OpenCV có thể không được ghi nhận đầy đủ.
import tracemalloc

# gc dùng để dọn rác trước mỗi lần đo, giúp kết quả đo ổn định hơn một chút.
import gc

# NumPy dùng để biểu diễn ảnh thành ma trận và tự cài đặt thuật toán xử lý pixel.
import numpy as np

# OpenCV dùng để đọc/ghi ảnh và làm phần đối chiếu bằng hàm thư viện.
import cv2

# Matplotlib dùng để hiển thị ảnh trong notebook.
import matplotlib.pyplot as plt

# Cấu hình để matplotlib hiển thị hình trực tiếp trong notebook.
%matplotlib inline

# Khai báo thư mục dữ liệu và thư mục lưu kết quả.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Đường dẫn ảnh tải bằng wget.
WGET_IMAGE_PATH = DATA_DIR / "Lenna.jpg"

# Đường dẫn ảnh dự phòng đi kèm bài nộp, dùng khi wget không tải được.
FALLBACK_IMAGE_PATH = DATA_DIR / "anh_mau_fallback.png"

# Hàm tạo ảnh dự phòng để notebook chạy được ngay cả khi không có Internet.
def tao_anh_du_phong(path):
    """Tạo ảnh màu đơn giản gồm gradient, hình khối và chữ để dễ quan sát phép biến đổi."""
    # Khai báo kích thước ảnh dự phòng.
    height, width = 256, 256

    # Tạo trục tọa độ chuẩn hóa từ 0 đến 1.
    x = np.linspace(0, 1, width, dtype=np.float32)
    y = np.linspace(0, 1, height, dtype=np.float32)

    # Tạo lưới tọa độ 2 chiều.
    xx, yy = np.meshgrid(x, y)

    # Sinh từng kênh màu theo định dạng BGR của OpenCV.
    blue = (50 + 180 * xx).astype(np.uint8)
    green = (70 + 160 * yy).astype(np.uint8)
    red = (160 + 70 * np.sin(2 * np.pi * (xx + yy))).clip(0, 255).astype(np.uint8)

    # Ghép ba kênh thành ảnh màu.
    image = np.dstack([blue, green, red])

    # Vẽ hình chữ nhật để quan sát biến dạng khi scale và xoay.
    cv2.rectangle(image, (20, 20), (110, 85), (255, 255, 255), 4)

    # Vẽ hình tròn để quan sát răng cưa và nội suy.
    cv2.circle(image, (190, 70), 45, (20, 20, 20), 5)

    # Vẽ tam giác để có vùng màu phẳng khi biến đổi màu.
    points = np.array([[55, 220], [130, 125], [210, 220]], dtype=np.int32)
    cv2.fillPoly(image, [points], (0, 220, 255))
    cv2.polylines(image, [points], True, (255, 255, 255), 2)

    # Vẽ các đường chéo mảnh để dễ thấy khác biệt nội suy.
    for i in range(0, width, 32):
        cv2.line(image, (i, 0), (0, i), (255, 255, 255), 1)

    # Ghi chữ TH01 để làm mốc khi xoay ảnh.
    cv2.putText(image, "TH01", (68, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(image, "TH01", (68, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 1, cv2.LINE_AA)

    # Lưu ảnh dự phòng ra file.
    cv2.imwrite(str(path), image)

# Nếu wget thất bại hoặc file tải về bị rỗng, tạo ảnh dự phòng.
if (not WGET_IMAGE_PATH.exists()) or WGET_IMAGE_PATH.stat().st_size == 0 or cv2.imread(str(WGET_IMAGE_PATH)) is None:
    print("Không đọc được ảnh Lenna từ wget, chuyển sang dùng ảnh dự phòng.")
    tao_anh_du_phong(FALLBACK_IMAGE_PATH)
    IMAGE_PATH = FALLBACK_IMAGE_PATH
else:
    print("Đọc được ảnh Lenna tải bằng wget.")
    IMAGE_PATH = WGET_IMAGE_PATH

# Đọc ảnh đầu vào bằng OpenCV. Ảnh màu của OpenCV có thứ tự kênh là BGR.
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)

# Nếu vẫn không đọc được ảnh thì dừng notebook với thông báo rõ ràng.
if image_bgr is None:
    raise FileNotFoundError(f"Không đọc được ảnh tại: {IMAGE_PATH}")

# In thông tin ảnh để kiểm tra nhanh.
print("Ảnh đang dùng:", IMAGE_PATH)
print("Kích thước ảnh:", image_bgr.shape)
print("Kiểu dữ liệu:", image_bgr.dtype)

# Hàm chuyển BGR sang RGB thủ công bằng đảo thứ tự kênh màu.
def bgr_to_rgb_manual(image):
    """Chuyển ảnh BGR sang RGB bằng slicing của NumPy."""
    return image[:, :, ::-1].copy()

# Hàm hiển thị nhiều ảnh cạnh nhau.
def show_images(images, titles, cols=3, figsize=(14, 4)):
    """Hiển thị danh sách ảnh với tiêu đề tương ứng."""
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for index, (img, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(rows, cols, index)
        if img.ndim == 2:
            plt.imshow(img, cmap="gray", vmin=0, vmax=255)
        else:
            plt.imshow(bgr_to_rgb_manual(img) if img.shape[-1] == 3 else img)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Hàm tính sai số bình phương trung bình giữa hai ảnh cùng kích thước.
def mse(image_a, image_b):
    """Tính Mean Squared Error giữa hai ảnh."""
    diff = image_a.astype(np.float32) - image_b.astype(np.float32)
    return float(np.mean(diff ** 2))

# Hàm đo thời gian chạy và bộ nhớ Python cấp phát của một hàm xử lý ảnh.
def measure_time_memory(func, *args, repeat=3, **kwargs):
    """Chạy hàm nhiều lần, trả về kết quả cuối, thời gian trung bình và peak memory."""
    times = []
    peaks_kb = []
    result = None

    for _ in range(repeat):
        # Dọn rác trước khi đo để giảm nhiễu.
        gc.collect()

        # Bắt đầu theo dõi bộ nhớ Python.
        tracemalloc.start()

        # Bắt đầu đo thời gian.
        start = time.perf_counter()

        # Gọi hàm cần đo.
        result = func(*args, **kwargs)

        # Tính thời gian đã chạy.
        elapsed = time.perf_counter() - start

        # Lấy bộ nhớ hiện tại và bộ nhớ đỉnh trong lúc chạy.
        current, peak = tracemalloc.get_traced_memory()

        # Dừng theo dõi bộ nhớ để tránh ảnh hưởng các cell sau.
        tracemalloc.stop()

        # Lưu kết quả đo.
        times.append(elapsed)
        peaks_kb.append(peak / 1024)

    # Trả về kết quả xử lý, thời gian trung bình và bộ nhớ đỉnh lớn nhất.
    return result, float(np.mean(times)), float(max(peaks_kb))

## 1. Đọc và hiển thị hình ảnh - open/show

Mục này có đúng 3 code cell: thủ công, tự động/thư viện và so sánh.

In [ ]:
# =========================
# CELL THỦ CÔNG - OPEN/SHOW
# =========================
# Ghi chú: JPEG decoding vẫn do cv2.imread đảm nhiệm vì tự viết bộ giải mã JPEG là ngoài phạm vi bài.
# Phần thủ công ở đây là tự chuyển kênh BGR sang RGB bằng thao tác mảng, không dùng cv2.cvtColor.

def open_show_manual_pipeline(path):
    """Đọc ảnh và tự chuyển BGR sang RGB để hiển thị bằng Matplotlib."""
    # Đọc ảnh từ đường dẫn bằng OpenCV.
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)

    # Kiểm tra lỗi đường dẫn hoặc file không hợp lệ.
    if img_bgr is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {path}")

    # Chuyển BGR sang RGB bằng slicing thủ công.
    img_rgb = img_bgr[:, :, ::-1].copy()

    # Trả về ảnh RGB để Matplotlib hiển thị đúng màu.
    return img_rgb

# Chạy pipeline thủ công một lần để lấy kết quả.
open_show_manual = open_show_manual_pipeline(IMAGE_PATH)

In [ ]:
# ==================================
# CELL TỰ ĐỘNG / THƯ VIỆN - OPEN/SHOW
# ==================================
# Ở phần này, ta dùng hàm cv2.cvtColor để chuyển BGR sang RGB.

def open_show_library_pipeline(path):
    """Đọc ảnh và chuyển BGR sang RGB bằng hàm thư viện OpenCV."""
    # Đọc ảnh từ đường dẫn bằng OpenCV.
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)

    # Kiểm tra ảnh có đọc thành công không.
    if img_bgr is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {path}")

    # Chuyển màu bằng hàm thư viện OpenCV.
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Trả về ảnh RGB để hiển thị bằng Matplotlib.
    return img_rgb

# Chạy pipeline thư viện một lần để lấy kết quả.
open_show_library = open_show_library_pipeline(IMAGE_PATH)

In [ ]:
# ===========================
# CELL SO SÁNH - OPEN/SHOW
# ===========================
# Đo thời gian và bộ nhớ cho cách thủ công.
open_show_manual_measured, t_open_manual, m_open_manual = measure_time_memory(
    open_show_manual_pipeline,
    IMAGE_PATH,
    repeat=5,
)

# Đo thời gian và bộ nhớ cho cách dùng thư viện.
open_show_library_measured, t_open_library, m_open_library = measure_time_memory(
    open_show_library_pipeline,
    IMAGE_PATH,
    repeat=5,
)

# Hiển thị hai kết quả để quan sát bằng mắt.
show_images(
    [open_show_manual_measured[:, :, ::-1], open_show_library_measured[:, :, ::-1]],
    ["Thủ công: BGR -> RGB bằng slicing", "Thư viện: cv2.cvtColor"],
    cols=2,
    figsize=(10, 4),
)

# In sai số và tài nguyên sử dụng.
print("MSE thủ công vs thư viện:", mse(open_show_manual_measured, open_show_library_measured))
print(f"Thời gian thủ công: {t_open_manual:.6f} giây | Peak memory: {m_open_manual:.2f} KB")
print(f"Thời gian thư viện: {t_open_library:.6f} giây | Peak memory: {m_open_library:.2f} KB")

## 2. Scale - thay đổi kích thước ảnh

Mục này dùng nội suy bilinear cho phần thủ công và `cv2.resize` cho phần thư viện.

In [ ]:
# =====================
# CELL THỦ CÔNG - SCALE
# =====================
# Tự cài đặt scale bằng nội suy song tuyến tính, không dùng cv2.resize.

def resize_bilinear_manual(image, new_width, new_height):
    """Thay đổi kích thước ảnh bằng nội suy bilinear thủ công."""
    # Lấy chiều cao và chiều rộng của ảnh nguồn.
    src_height, src_width = image.shape[:2]

    # Tạo ảnh đầu ra có kích thước mới.
    if image.ndim == 2:
        output = np.zeros((new_height, new_width), dtype=np.uint8)
    else:
        output = np.zeros((new_height, new_width, image.shape[2]), dtype=np.uint8)

    # Tính tỉ lệ ánh xạ tọa độ từ ảnh đầu ra về ảnh nguồn.
    x_ratio = 0 if new_width == 1 else (src_width - 1) / (new_width - 1)
    y_ratio = 0 if new_height == 1 else (src_height - 1) / (new_height - 1)

    # Duyệt từng pixel của ảnh đầu ra.
    for y_out in range(new_height):
        # Tọa độ thực theo trục y trên ảnh nguồn.
        y = y_out * y_ratio
        y0 = int(np.floor(y))
        y1 = min(y0 + 1, src_height - 1)
        wy = y - y0

        for x_out in range(new_width):
            # Tọa độ thực theo trục x trên ảnh nguồn.
            x = x_out * x_ratio
            x0 = int(np.floor(x))
            x1 = min(x0 + 1, src_width - 1)
            wx = x - x0

            # Nội suy theo chiều ngang ở hàng trên.
            top = (1 - wx) * image[y0, x0] + wx * image[y0, x1]

            # Nội suy theo chiều ngang ở hàng dưới.
            bottom = (1 - wx) * image[y1, x0] + wx * image[y1, x1]

            # Nội suy tiếp theo chiều dọc.
            value = (1 - wy) * top + wy * bottom

            # Chặn giá trị về [0, 255] và ép kiểu uint8.
            output[y_out, x_out] = np.clip(value, 0, 255).astype(np.uint8)

    # Trả về ảnh đã resize.
    return output

# Kích thước mới dùng trong bài.
SCALE_SIZE = (384, 384)

# Chạy scale thủ công một lần.
scale_manual = resize_bilinear_manual(image_bgr, SCALE_SIZE[0], SCALE_SIZE[1])

In [ ]:
# ================================
# CELL TỰ ĐỘNG / THƯ VIỆN - SCALE
# ================================
# Dùng cv2.resize với INTER_LINEAR để tương ứng với bilinear interpolation.

def resize_library(image, new_width, new_height):
    """Thay đổi kích thước ảnh bằng hàm cv2.resize."""
    # cv2.resize nhận kích thước theo thứ tự (width, height).
    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)

# Chạy scale bằng thư viện một lần.
scale_library = resize_library(image_bgr, SCALE_SIZE[0], SCALE_SIZE[1])

In [ ]:
# ========================
# CELL SO SÁNH - SCALE
# ========================
# Đo thời gian và bộ nhớ cho scale thủ công.
scale_manual_measured, t_scale_manual, m_scale_manual = measure_time_memory(
    resize_bilinear_manual,
    image_bgr,
    SCALE_SIZE[0],
    SCALE_SIZE[1],
    repeat=1,
)

# Đo thời gian và bộ nhớ cho scale thư viện.
scale_library_measured, t_scale_library, m_scale_library = measure_time_memory(
    resize_library,
    image_bgr,
    SCALE_SIZE[0],
    SCALE_SIZE[1],
    repeat=5,
)

# Tạo ảnh sai khác tuyệt đối để nhìn vùng khác nhau.
scale_diff = cv2.absdiff(scale_manual_measured, scale_library_measured)

# Hiển thị ảnh gốc, kết quả thủ công, kết quả thư viện và sai khác.
show_images(
    [image_bgr, scale_manual_measured, scale_library_measured, scale_diff],
    ["Ảnh gốc", "Scale thủ công", "Scale cv2.resize", "Sai khác tuyệt đối"],
    cols=4,
    figsize=(16, 4),
)

# In sai số, thời gian và bộ nhớ.
print("MSE scale thủ công vs thư viện:", mse(scale_manual_measured, scale_library_measured))
print(f"Thời gian thủ công: {t_scale_manual:.6f} giây | Peak memory: {m_scale_manual:.2f} KB")
print(f"Thời gian thư viện: {t_scale_library:.6f} giây | Peak memory: {m_scale_library:.2f} KB")

## 3. Xoay ảnh

Mục này tự xoay ảnh bằng ánh xạ ngược và so sánh với `cv2.warpAffine`.

In [ ]:
# ====================
# CELL THỦ CÔNG - XOAY
# ====================
# Tự cài đặt xoay ảnh quanh tâm bằng ánh xạ ngược và nearest neighbor.

def rotate_nearest_manual(image, angle_degrees, fill_color=(255, 255, 255)):
    """Xoay ảnh quanh tâm, giữ nguyên kích thước đầu ra."""
    # Lấy kích thước ảnh nguồn.
    height, width = image.shape[:2]

    # Tạo ảnh đầu ra với nền trắng.
    if image.ndim == 2:
        output = np.full((height, width), 255, dtype=np.uint8)
    else:
        output = np.full((height, width, image.shape[2]), fill_color, dtype=np.uint8)

    # Đổi góc từ độ sang radian.
    angle = np.deg2rad(angle_degrees)
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)

    # Tâm ảnh là điểm xoay.
    center_x = (width - 1) / 2.0
    center_y = (height - 1) / 2.0

    # Duyệt từng pixel trên ảnh đầu ra.
    for y_out in range(height):
        for x_out in range(width):
            # Dịch tọa độ về hệ trục có gốc tại tâm ảnh.
            x_shift = x_out - center_x
            y_shift = y_out - center_y

            # Ánh xạ ngược từ ảnh đầu ra về ảnh nguồn.
            x_src = cos_a * x_shift + sin_a * y_shift + center_x
            y_src = -sin_a * x_shift + cos_a * y_shift + center_y

            # Lấy pixel gần nhất.
            x_round = int(round(x_src))
            y_round = int(round(y_src))

            # Nếu tọa độ nguồn nằm trong ảnh, sao chép pixel đó.
            if 0 <= x_round < width and 0 <= y_round < height:
                output[y_out, x_out] = image[y_round, x_round]

    # Trả về ảnh đã xoay.
    return output

# Góc xoay dùng để kiểm thử.
ROTATE_ANGLE = 30

# Chạy xoay thủ công một lần.
rotate_manual = rotate_nearest_manual(image_bgr, ROTATE_ANGLE)

In [ ]:
# ===============================
# CELL TỰ ĐỘNG / THƯ VIỆN - XOAY
# ===============================
# Dùng cv2.getRotationMatrix2D và cv2.warpAffine để xoay ảnh.

def rotate_library(image, angle_degrees):
    """Xoay ảnh bằng OpenCV, giữ nguyên kích thước ảnh."""
    # Lấy kích thước ảnh.
    height, width = image.shape[:2]

    # Tính tâm xoay theo định dạng (x, y).
    center = ((width - 1) / 2.0, (height - 1) / 2.0)

    # Tạo ma trận affine cho phép xoay quanh tâm.
    matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)

    # Dùng INTER_NEAREST để tương ứng với phần thủ công.
    return cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_NEAREST,
        borderValue=(255, 255, 255),
    )

# Chạy xoay bằng thư viện một lần.
rotate_library_result = rotate_library(image_bgr, ROTATE_ANGLE)

In [ ]:
# =======================
# CELL SO SÁNH - XOAY
# =======================
# Đo thời gian và bộ nhớ cho xoay thủ công.
rotate_manual_measured, t_rotate_manual, m_rotate_manual = measure_time_memory(
    rotate_nearest_manual,
    image_bgr,
    ROTATE_ANGLE,
    repeat=1,
)

# Đo thời gian và bộ nhớ cho xoay thư viện.
rotate_library_measured, t_rotate_library, m_rotate_library = measure_time_memory(
    rotate_library,
    image_bgr,
    ROTATE_ANGLE,
    repeat=5,
)

# Tạo ảnh sai khác tuyệt đối.
rotate_diff = cv2.absdiff(rotate_manual_measured, rotate_library_measured)

# Hiển thị kết quả.
show_images(
    [image_bgr, rotate_manual_measured, rotate_library_measured, rotate_diff],
    ["Ảnh gốc", "Xoay thủ công", "Xoay cv2.warpAffine", "Sai khác tuyệt đối"],
    cols=4,
    figsize=(16, 4),
)

# In sai số và tài nguyên.
print("MSE xoay thủ công vs thư viện:", mse(rotate_manual_measured, rotate_library_measured))
print(f"Thời gian thủ công: {t_rotate_manual:.6f} giây | Peak memory: {m_rotate_manual:.2f} KB")
print(f"Thời gian thư viện: {t_rotate_library:.6f} giây | Peak memory: {m_rotate_library:.2f} KB")

## 4. Biến đổi màu sắc

Mục này gồm chuyển ảnh sang grayscale và tạo ảnh âm bản.

In [ ]:
# ==========================
# CELL THỦ CÔNG - MÀU SẮC
# ==========================
# Tự chuyển grayscale bằng công thức độ chói và tự tạo ảnh âm bản.

def grayscale_manual(image):
    """Chuyển ảnh BGR sang ảnh xám bằng công thức Y = 0.114B + 0.587G + 0.299R."""
    # Tách từng kênh màu B, G, R và chuyển sang float để tính toán chính xác.
    blue = image[:, :, 0].astype(np.float32)
    green = image[:, :, 1].astype(np.float32)
    red = image[:, :, 2].astype(np.float32)

    # Tính giá trị xám theo trọng số cảm nhận thị giác.
    gray = 0.114 * blue + 0.587 * green + 0.299 * red

    # Chặn giá trị và chuyển về uint8.
    return np.clip(gray, 0, 255).astype(np.uint8)

def negative_manual(image):
    """Tạo ảnh âm bản bằng công thức I' = 255 - I."""
    # Với ảnh uint8, mỗi pixel nằm trong [0, 255].
    return 255 - image

# Chạy biến đổi màu thủ công một lần.
gray_manual = grayscale_manual(image_bgr)
negative_manual_result = negative_manual(image_bgr)

In [ ]:
# ======================================
# CELL TỰ ĐỘNG / THƯ VIỆN - MÀU SẮC
# ======================================
# Dùng hàm OpenCV để chuyển xám và tạo ảnh âm bản.

def color_library(image):
    """Chuyển xám và tạo ảnh âm bản bằng hàm thư viện OpenCV."""
    # Chuyển ảnh BGR sang grayscale bằng cv2.cvtColor.
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Tạo ảnh âm bản bằng cv2.bitwise_not.
    negative = cv2.bitwise_not(image)

    # Trả về cả hai kết quả để tiện so sánh.
    return gray, negative

# Chạy biến đổi màu bằng thư viện một lần.
gray_library, negative_library = color_library(image_bgr)

In [ ]:
# ===========================
# CELL SO SÁNH - MÀU SẮC
# ===========================
# Đo thời gian và bộ nhớ cho phần grayscale thủ công.
gray_manual_measured, t_gray_manual, m_gray_manual = measure_time_memory(
    grayscale_manual,
    image_bgr,
    repeat=5,
)

# Đo thời gian và bộ nhớ cho grayscale thư viện.
gray_library_measured, t_gray_library, m_gray_library = measure_time_memory(
    lambda img: cv2.cvtColor(img, cv2.COLOR_BGR2GRAY),
    image_bgr,
    repeat=5,
)

# Đo thời gian và bộ nhớ cho âm bản thủ công.
negative_manual_measured, t_neg_manual, m_neg_manual = measure_time_memory(
    negative_manual,
    image_bgr,
    repeat=5,
)

# Đo thời gian và bộ nhớ cho âm bản thư viện.
negative_library_measured, t_neg_library, m_neg_library = measure_time_memory(
    cv2.bitwise_not,
    image_bgr,
    repeat=5,
)

# Hiển thị kết quả.
show_images(
    [image_bgr, gray_manual_measured, gray_library_measured, negative_manual_measured, negative_library_measured],
    ["Ảnh gốc", "Xám thủ công", "Xám cv2", "Âm bản thủ công", "Âm bản cv2"],
    cols=5,
    figsize=(18, 4),
)

# In sai số và tài nguyên.
print("MSE grayscale thủ công vs thư viện:", mse(gray_manual_measured, gray_library_measured))
print("MSE negative thủ công vs thư viện:", mse(negative_manual_measured, negative_library_measured))
print(f"Grayscale thủ công: {t_gray_manual:.6f} giây | Peak memory: {m_gray_manual:.2f} KB")
print(f"Grayscale thư viện: {t_gray_library:.6f} giây | Peak memory: {m_gray_library:.2f} KB")
print(f"Negative thủ công:  {t_neg_manual:.6f} giây | Peak memory: {m_neg_manual:.2f} KB")
print(f"Negative thư viện:  {t_neg_library:.6f} giây | Peak memory: {m_neg_library:.2f} KB")

## 5. Biến đổi độ sáng và độ tương phản

Mục này dùng công thức tuyến tính `I' = alpha * I + beta` và so sánh với `cv2.convertScaleAbs`.

In [ ]:
# ===============================================
# CELL THỦ CÔNG - ĐỘ SÁNG VÀ ĐỘ TƯƠNG PHẢN
# ===============================================
# Tự áp dụng công thức I' = alpha * I + beta trên từng pixel.

def brightness_contrast_manual(image, alpha=1.25, beta=35):
    """Chỉnh độ tương phản bằng alpha và độ sáng bằng beta."""
    # Chuyển sang float32 để phép nhân/cộng không bị tràn số uint8.
    image_float = image.astype(np.float32)

    # Áp dụng biến đổi tuyến tính cho từng pixel.
    transformed = alpha * image_float + beta

    # Chặn về [0, 255] rồi chuyển về uint8.
    return np.clip(transformed, 0, 255).astype(np.uint8)

# Hệ số tương phản và độ sáng.
ALPHA = 1.25
BETA = 35

# Chạy chỉnh sáng/tương phản thủ công một lần.
bc_manual = brightness_contrast_manual(image_bgr, alpha=ALPHA, beta=BETA)

In [ ]:
# ===========================================================
# CELL TỰ ĐỘNG / THƯ VIỆN - ĐỘ SÁNG VÀ ĐỘ TƯƠNG PHẢN
# ===========================================================
# Dùng cv2.convertScaleAbs để OpenCV tự xử lý nhân alpha, cộng beta và ép kiểu.

def brightness_contrast_library(image, alpha=1.25, beta=35):
    """Chỉnh sáng/tương phản bằng hàm OpenCV."""
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

# Chạy bằng thư viện một lần.
bc_library = brightness_contrast_library(image_bgr, alpha=ALPHA, beta=BETA)

In [ ]:
# ================================================
# CELL SO SÁNH - ĐỘ SÁNG VÀ ĐỘ TƯƠNG PHẢN
# ================================================
# Đo thời gian và bộ nhớ cho cách thủ công.
bc_manual_measured, t_bc_manual, m_bc_manual = measure_time_memory(
    brightness_contrast_manual,
    image_bgr,
    alpha=ALPHA,
    beta=BETA,
    repeat=5,
)

# Đo thời gian và bộ nhớ cho cách dùng thư viện.
bc_library_measured, t_bc_library, m_bc_library = measure_time_memory(
    brightness_contrast_library,
    image_bgr,
    alpha=ALPHA,
    beta=BETA,
    repeat=5,
)

# Tính ảnh sai khác tuyệt đối.
bc_diff = cv2.absdiff(bc_manual_measured, bc_library_measured)

# Hiển thị ảnh.
show_images(
    [image_bgr, bc_manual_measured, bc_library_measured, bc_diff],
    ["Ảnh gốc", "Thủ công: alpha*I + beta", "cv2.convertScaleAbs", "Sai khác tuyệt đối"],
    cols=4,
    figsize=(16, 4),
)

# In sai số và tài nguyên.
print("MSE chỉnh sáng/tương phản thủ công vs thư viện:", mse(bc_manual_measured, bc_library_measured))
print(f"Thời gian thủ công: {t_bc_manual:.6f} giây | Peak memory: {m_bc_manual:.2f} KB")
print(f"Thời gian thư viện: {t_bc_library:.6f} giây | Peak memory: {m_bc_library:.2f} KB")

## 6. Lưu kết quả minh chứng

Cell này không phải mục xử lý chính. Nó chỉ lưu một số ảnh kết quả ra thư mục `outputs/` để tiện kiểm tra khi nộp bài.

In [ ]:
# Lưu ảnh scale bằng thủ công và thư viện.
cv2.imwrite(str(OUTPUT_DIR / "scale_manual.png"), scale_manual)
cv2.imwrite(str(OUTPUT_DIR / "scale_library.png"), scale_library)

# Lưu ảnh xoay bằng thủ công và thư viện.
cv2.imwrite(str(OUTPUT_DIR / "rotate_manual.png"), rotate_manual)
cv2.imwrite(str(OUTPUT_DIR / "rotate_library.png"), rotate_library_result)

# Lưu ảnh biến đổi màu và chỉnh sáng/tương phản.
cv2.imwrite(str(OUTPUT_DIR / "gray_manual.png"), gray_manual)
cv2.imwrite(str(OUTPUT_DIR / "negative_manual.png"), negative_manual_result)
cv2.imwrite(str(OUTPUT_DIR / "brightness_contrast_manual.png"), bc_manual)

# In danh sách file kết quả đã lưu.
for path in sorted(OUTPUT_DIR.glob("*.png")):
    print(path)

## 7. Kết luận

- Cell thủ công giúp hiểu bản chất: thao tác kênh màu, nội suy, ánh xạ tọa độ và biến đổi tuyến tính pixel.
- Cell thư viện cho thấy cách làm ngắn gọn, ổn định và nhanh hơn trong thực tế.
- Cell so sánh giúp kiểm tra bằng mắt, bằng MSE, bằng thời gian chạy và bằng bộ nhớ xấp xỉ.